In [1]:
import sys
from pathlib import Path

# Add the project root directory to Python's import path
project_root = Path.cwd().resolve().parent
sys.path.insert(0, str(project_root))

# Import our preprocessing function
from src.preprocessing import tokenize_and_align_labels

print("Preprocessing function imported successfully!")

Preprocessing function imported successfully!


In [2]:
from datasets import load_dataset
from transformers import AutoTokenizer

# Load the CoNLL-2003 dataset
dataset = load_dataset("lhoestq/conll2003")

# Load the BERT tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

# Select one training example
sample = dataset["train"][0]

sample_batch = {
    "tokens": [sample["tokens"]],
    "ner_tags": [sample["ner_tags"]]
}

# Call the imported preprocessing function
processed_sample = tokenize_and_align_labels(
    sample_batch,
    tokenizer
)

# Verify the result
print("Input IDs:", processed_sample["input_ids"][0])
print("Aligned Labels:", processed_sample["labels"][0])

print("Token count:", len(processed_sample["input_ids"][0]))
print("Label count:", len(processed_sample["labels"][0]))

Input IDs: [101, 7270, 22961, 1528, 1840, 1106, 21423, 1418, 2495, 12913, 119, 102]
Aligned Labels: [-100, 3, 0, 7, 0, 0, 0, 7, 0, -100, 0, -100]
Token count: 12
Label count: 12


In [3]:
from functools import partial

# Connect our preprocessing function with the tokenizer
preprocessing_function = partial(
    tokenize_and_align_labels,
    tokenizer=tokenizer
)

# Apply preprocessing to all dataset splits
tokenized_dataset = dataset.map(
    preprocessing_function,
    batched=True
)

# Display results
print(tokenized_dataset)

# Verify all examples
for split in ["train", "validation", "test"]:
    mismatches = sum(
        len(example["input_ids"]) != len(example["labels"])
        for example in tokenized_dataset[split]
    )

    print(f"{split}: {mismatches} mismatched examples")

Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 3453
    })
})
train: 0 mismatched examples
validation: 0 mismatched examples
test: 0 mismatched examples
